In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Load Data
df=pd.read_csv("nassau_candy.csv")
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Country/Region,City,State/Province,Postal Code,Division,Region,Product ID,Product Name,Sales,Units,Gross Profit,Cost
0,1,US-2021-103800-CHO-MIL-31000,03-01-2024,30-06-2026,Standard Class,103800,United States,Houston,Texas,77095,Chocolate,Interior,CHO-MIL-31000,Wonka Bar - Milk Chocolate,6.50,2,4.22,2.28
1,2,US-2021-112326-CHO-TRI-54000,04-01-2024,01-07-2026,Standard Class,112326,United States,Naperville,Illinois,60540,Chocolate,Interior,CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,7.50,2,4.90,2.60
2,3,US-2021-112326-CHO-NUT-13000,04-01-2024,01-07-2026,Standard Class,112326,United States,Naperville,Illinois,60540,Chocolate,Interior,CHO-NUT-13000,Wonka Bar - Nutty Crunch Surprise,10.47,3,7.47,3.00
3,4,US-2021-112326-CHO-SCR-58000,04-01-2024,01-07-2026,Standard Class,112326,United States,Naperville,Illinois,60540,Chocolate,Interior,CHO-SCR-58000,Wonka Bar -Scrumdiddlyumptious,10.80,3,7.50,3.30
4,5,US-2021-141817-CHO-TRI-54000,05-01-2024,05-07-2026,Standard Class,141817,United States,Philadelphia,Pennsylvania,19143,Chocolate,Atlantic,CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,11.25,3,7.35,3.90


In [3]:
# Inspect data types
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10194 entries, 0 to 10193
Data columns (total 18 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Row ID          10194 non-null  int64  
 1   Order ID        10194 non-null  str    
 2   Order Date      10194 non-null  str    
 3   Ship Date       10194 non-null  str    
 4   Ship Mode       10194 non-null  str    
 5   Customer ID     10194 non-null  int64  
 6   Country/Region  10194 non-null  str    
 7   City            10194 non-null  str    
 8   State/Province  10194 non-null  str    
 9   Postal Code     10194 non-null  str    
 10  Division        10194 non-null  str    
 11  Region          10194 non-null  str    
 12  Product ID      10194 non-null  str    
 13  Product Name    10194 non-null  str    
 14  Sales           10194 non-null  float64
 15  Units           10194 non-null  int64  
 16  Gross Profit    10194 non-null  float64
 17  Cost            10194 non-null  float64
dt

In [4]:
# Missing Values
missing=df.isna().sum().sort_values(ascending=False)
print(missing)

Row ID            0
Order ID          0
Order Date        0
Ship Date         0
Ship Mode         0
Customer ID       0
Country/Region    0
City              0
State/Province    0
Postal Code       0
Division          0
Region            0
Product ID        0
Product Name      0
Sales             0
Units             0
Gross Profit      0
Cost              0
dtype: int64


In [5]:
# Exact duplicates
print("Duplicate rows:",df.duplicated().sum())

Duplicate rows: 0


In [6]:
# Unique Values
for col in ["Ship Mode", "Region","Division","Country/Region"]: 
    print(f"\n{col}")
print(df[col].value_counts(dropna=False))


Ship Mode

Region

Division

Country/Region
Country/Region
United States    9994
Canada            200
Name: count, dtype: int64


In [7]:
# Date Investigation
df["Order_Date_Test"]=pd.to_datetime(
    df["Order Date"],
    dayfirst=True,
    errors="coerce"
)
df["Ship_Date_Test"]=pd.to_datetime(
    df["Ship Date"],
    dayfirst=True,
    errors="coerce"
)

In [8]:
print(df["Order_Date_Test"].min())
print(df["Order_Date_Test"].max())

print(df["Ship_Date_Test"].min())
print(df["Ship_Date_Test"].max())

2024-01-02 00:00:00
2025-12-31 00:00:00
2026-06-30 00:00:00
2030-06-28 00:00:00


In [9]:
df["Preliminary_Lead_Time"]=(
    df["Ship_Date_Test"]-df["Order_Date_Test"]
).dt.days

In [10]:
df["Preliminary_Lead_Time"].describe()

count    10194.000000
mean      1320.841868
std        262.444892
min        904.000000
25%       1271.000000
50%       1274.000000
75%       1638.000000
max       1642.000000
Name: Preliminary_Lead_Time, dtype: float64

In [11]:
# Inspect the Actual Problem
df[
    [
        "Order ID",
        "Order Date",
        "Ship Date",
        "Order_Date_Test",
        "Ship_Date_Test",
        "Preliminary_Lead_Time"
    ]
].head(20)

,Order ID,Order Date,Ship Date,Order_Date_Test,Ship_Date_Test,Preliminary_Lead_Time
0,US-2021-103800-CHO-MIL-31000,03-01-2024,30-06-2026,2024-01-03,2026-06-30,909
1,US-2021-112326-CHO-TRI-54000,04-01-2024,01-07-2026,2024-01-04,2026-07-01,909
2,US-2021-112326-CHO-NUT-13000,04-01-2024,01-07-2026,2024-01-04,2026-07-01,909
3,US-2021-112326-CHO-SCR-58000,04-01-2024,01-07-2026,2024-01-04,2026-07-01,909
4,US-2021-141817-CHO-TRI-54000,05-01-2024,05-07-2026,2024-01-05,2026-07-05,912
5,US-2021-167199-CHO-SCR-58000,06-01-2024,03-07-2026,2024-01-06,2026-07-03,909
6,US-2021-167199-CHO-TRI-54000,06-01-2024,03-07-2026,2024-01-06,2026-07-03,909
7,US-2021-106054-CHO-MIL-31000,06-01-2024,30-06-2026,2024-01-06,2026-06-30,906
8,US-2021-167199-CHO-NUT-13000,06-01-2024,03-07-2026,2024-01-06,2026-07-03,909
9,US-2021-167199-CHO-MIL-31000,06-01-2024,03-07-2026,2024-01-06,2026-07-03,909


In [12]:
df.sort_values(
    "Preliminary_Lead_Time",
    ascending=False
)[
    [
        "Order ID",
        "Order Date",
        "Ship Date",
        "Preliminary_Lead_Time"
    ]
].head(20)

,Order ID,Order Date,Ship Date,Preliminary_Lead_Time
10183,CA-2024-146626-CHO-SCR-58000,29-12-2025,28-06-2030,1642
10175,US-2024-146626-CHO-SCR-58000,29-12-2025,28-06-2030,1642
7167,US-2024-105851-CHO-NUT-13000,13-03-2025,10-09-2029,1642
9540,US-2024-139969-CHO-MIL-31000,19-11-2025,19-05-2030,1642
10165,US-2024-164826-CHO-TRI-54000,28-12-2025,27-06-2030,1642
10164,US-2024-164826-CHO-FUD-51000,28-12-2025,27-06-2030,1642
10160,US-2024-164826-CHO-SCR-58000,28-12-2025,27-06-2030,1642
8087,US-2024-108560-CHO-SCR-58000,08-07-2025,05-01-2030,1642
10120,US-2024-147956-CHO-SCR-58000,24-12-2025,23-06-2030,1642
10111,US-2024-128734-CHO-MIL-31000,24-12-2025,23-06-2030,1642


In [13]:
# Financial Validation 
df["Profit_Check"]=(
    df["Sales"]
    - df["Cost"]
    - df["Gross Profit"]
)

print(
    "Profit mismatches.",
    (df["Profit_Check"].abs()>0.01).sum()
)

Profit mismatches. 0


In [21]:
print("Sales:",df["Sales"].sum())
print("Units:",df["Units"].sum())
print("Gross Profit:",df["Gross Profit"].sum())
print("Cost:",df["Cost"].sum())

Sales: 141783.63
Units: 38654
Gross Profit: 93442.79999999999
Cost: 48340.83
